In [ ]:
!pip install geopandas hdbscan xgboost

# Tahap 1 Setup, import library, dan baca data

In [ ]:
import os, joblib
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import hdbscan
from shapely.geometry import Point
from scipy.stats import randint, uniform, loguniform
from sklearn.neighbors import BallTree
from sklearn.metrics.pairwise import haversine_distances
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, precision_recall_curve,
                             confusion_matrix, roc_curve, ConfusionMatrixDisplay)
from xgboost import XGBClassifier

# ---------------- konfigurasi ----------------
BATAS_TRAIN = "2024-12-31"
RS = 42
STASIUN = {                      # verifikasi koordinat ini dengan data NASA POWER
    "Palangka Raya": (-2.21, 113.92),
    "Sampit":        (-2.53, 112.95),
    "Pangkalan Bun": (-2.68, 111.62),
    "Muara Teweh":   (-0.95, 114.89),
    "Kuala Kapuas":  (-3.00, 114.39),
}
VAR = ["T2M", "RH2M", "PRECTOTCORR", "WS10M"]

# ---------------- cache ----------------
PAKAI_DRIVE = True      # False kalau tidak mau menyentuh Google Drive
if PAKAI_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = "/content/drive/MyDrive/hotspot_kalteng"
else:
    SAVE_DIR = "/content/cache"
os.makedirs(SAVE_DIR, exist_ok=True)

def cached(nama, fn):
    """Kalau file cache ada -> load; kalau belum -> hitung, simpan, lalu kembalikan."""
    path = f"{SAVE_DIR}/{nama}.joblib"
    if os.path.exists(path):
        print("[cache] load:", nama)
        return joblib.load(path)
    hasil = fn()
    joblib.dump(hasil, path)
    print("[cache] simpan:", nama)
    return hasil

# ---------------- baca data ----------------
df_hotspot = pd.read_csv("/content/hotspot_kalimantan_tengah.csv")
df_meteo   = pd.read_csv("/content/meteorologi_kalimantan_tengah.csv")
batas      = gpd.read_file("/content/Kalimantan Tengah.geojson")
print("Ukuran dataset hotspot:", df_hotspot.shape)
print("Ukuran dataset meteorologi:", df_meteo.shape)

# Tahap 2 Cleaning data meteorologi

In [ ]:
# salin dari df_meteo dan ubah kolom tanggal ke datetime
met = df_meteo.copy()
met["tanggal"] = pd.to_datetime(met["tanggal"])

met = met.drop_duplicates(["lokasi", "tanggal"]).sort_values(["lokasi", "tanggal"])

# -999 = missing value NASA POWER
met[VAR] = met[VAR].replace(-999, np.nan)
print("Missing sebelum diisi:\n", met[VAR].isna().sum())

# isi missing dengan interpolasi per stasiun
met[VAR] = met.groupby("lokasi")[VAR].transform(
    lambda s: s.interpolate(limit_direction="both"))
print("Missing sesudah diisi:", met[VAR].isna().sum().sum())

# fitur turunan (rolling per stasiun)
g = met.groupby("lokasi")
met["CH_3d"]  = g["PRECTOTCORR"].transform(lambda s: s.rolling(3,  min_periods=1).sum())
met["CH_7d"]  = g["PRECTOTCORR"].transform(lambda s: s.rolling(7,  min_periods=1).sum())
met["CH_14d"] = g["PRECTOTCORR"].transform(lambda s: s.rolling(14, min_periods=1).sum())
met["RH_7d"]  = g["RH2M"].transform(lambda s: s.rolling(7, min_periods=1).mean())
met["T_7d"]   = g["T2M"].transform(lambda s: s.rolling(7, min_periods=1).mean())

def hari_kering_berturut(s, ambang=1.0):
    kering = s < ambang
    grup = (~kering).cumsum()
    return kering.groupby(grup).cumsum()

met["HKB"] = g["PRECTOTCORR"].transform(hari_kering_berturut)  # hari kering berturut-turut

# Tahap 3 Cleaning data hotspot dan pemotongan batas provinsi

In [ ]:
hs = df_hotspot.copy()
hs["acq_date"] = pd.to_datetime(hs["acq_date"])
print("Jumlah awal:", len(hs))

# buang deteksi confidence rendah ('l'); hapus baris ini kalau mau semua dipakai
hs = hs[hs["confidence"].isin(["n", "h"])]
hs = hs.rename(columns={"acq_date": "tanggal"})
hs = hs[["tanggal", "latitude", "longitude"]].dropna().reset_index(drop=True)
print("Setelah filter confidence:", len(hs))

# ---- potong ke batas provinsi ----
batas = batas.to_crs("EPSG:4326")
try:
    batas_union = batas.geometry.union_all()      # geopandas >= 1.0
except AttributeError:
    batas_union = batas.geometry.unary_union      # geopandas lama

gdf_hs = gpd.GeoDataFrame(
    hs.copy(),
    geometry=gpd.points_from_xy(hs["longitude"], hs["latitude"]),
    crs="EPSG:4326",
)
dalam = gdf_hs.geometry.intersects(batas_union)
print("Hotspot dalam provinsi :", dalam.sum())
print("Hotspot di luar        :", (~dalam).sum())

hs = gdf_hs.loc[dalam].drop(columns="geometry").reset_index(drop=True)

# ---- cek posisi stasiun terhadap batas provinsi ----
for nm, (la, lo) in STASIUN.items():
    print(f"{nm:15s} di dalam provinsi: {batas_union.contains(Point(lo, la))}")

# Tahap 4 Fungsi pendukung dan parameter awal

In [ ]:
MS = 30    # min_samples HDBSCAN

def evaluasi(nama, y, proba, thr=0.5, **extra):
    pred = (np.asarray(proba) >= thr).astype(int)
    d = {"Model": nama,
         "Accuracy":  accuracy_score(y, pred),
         "Precision": precision_score(y, pred, zero_division=0),
         "Recall":    recall_score(y, pred, zero_division=0),
         "F1-score":  f1_score(y, pred, zero_division=0),
         "ROC-AUC":   roc_auc_score(y, proba),
         "PR-AUC":    average_precision_score(y, proba)}
    d.update(extra)
    return d

def thr_optimal(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1s = 2 * prec * rec / (prec + rec + 1e-12)
    return float(thr[np.argmax(f1s[:-1])])

# ---------- zonasi ----------
def zona_hdbscan(h, mt, selection="leaf", mcs=150, ms=MS):
    Xtr = np.radians(h.loc[mt, ["latitude", "longitude"]].values)
    Xte = np.radians(h.loc[~mt, ["latitude", "longitude"]].values)
    cl = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=ms, metric="haversine",
                         cluster_selection_method=selection, prediction_data=True).fit(Xtr)
    lab_te, _ = hdbscan.approximate_predict(cl, Xte)
    return cl.labels_, lab_te

def zona_seragam(fn_zona, R_km=5.0):
    """Titik test -> zona training terdekat dalam R km (aturan penugasan yang seragam)."""
    def f(h, mt, **kw):
        lab_tr, _ = fn_zona(h, mt, **kw)
        Xtr = np.radians(h.loc[mt,  ["latitude", "longitude"]].values)
        Xte = np.radians(h.loc[~mt, ["latitude", "longitude"]].values)
        ok = lab_tr >= 0
        d, idx = BallTree(Xtr[ok], metric="haversine").query(Xte, k=1)
        lab_te = np.where(d[:, 0] * 6371 <= R_km, lab_tr[ok][idx[:, 0]], -1)
        return lab_tr, lab_te
    return f

# ---------- parameter awal (hasil tuning v1, supaya tidak perlu mengulang 50 menit) ----------
RF_PARAMS = {'max_depth': 5, 'max_features': 0.5, 'min_samples_leaf': 3,
             'min_samples_split': 18, 'n_estimators': 591}
XGB_PARAMS = {'colsample_bytree': 0.908897907718663, 'gamma': 0.0993578407670862,
              'learning_rate': 0.011049202253484457, 'max_depth': 5, 'min_child_weight': 3,
              'n_estimators': 280, 'subsample': 0.88453678109946}

# ---------- model (membaca RF_PARAMS / XGB_PARAMS global saat dipanggil) ----------
def buat_rf(seed=RS, cw=None):
    return RandomForestClassifier(**RF_PARAMS, class_weight=cw, random_state=seed, n_jobs=-1)

def buat_xgb(seed=RS, spw=1.0):
    return XGBClassifier(**XGB_PARAMS, objective="binary:logistic", eval_metric="logloss",
                         scale_pos_weight=spw, tree_method="hist", random_state=seed, n_jobs=-1)

def latih(X, y, skenario="tanpa"):
    if skenario == "class_weight":
        spw = (y == 0).sum() / (y == 1).sum()
        return buat_rf(cw="balanced_subsample").fit(X, y), buat_xgb(spw=spw).fit(X, y)
    return buat_rf().fit(X, y), buat_xgb().fit(X, y)

def proba_ens(rf, xgb, X, w=0.5):
    return w * rf.predict_proba(X)[:, 1] + (1 - w) * xgb.predict_proba(X)[:, 1]

# TAHAP 5 Zonasi spasial HDBSCAN, fitur cuaca tambahan, dan visualisasi peta

In [ ]:
from scipy.stats import loguniform

def tambah_fitur_met(m):
    m = m.sort_values(["lokasi", "tanggal"]).copy()
    es = 0.6108 * np.exp(17.27 * m["T2M"] / (m["T2M"] + 237.3))     # tekanan uap jenuh (kPa)
    m["VPD"] = es * (1 - m["RH2M"].clip(0, 100) / 100)
    g = m.groupby("lokasi")
    m["VPD_7d"] = g["VPD"].transform(lambda s: s.rolling(7, min_periods=1).mean())
    m["CH_30d"] = g["PRECTOTCORR"].transform(lambda s: s.rolling(30, min_periods=1).sum())
    m["CH_60d"] = g["PRECTOTCORR"].transform(lambda s: s.rolling(60, min_periods=1).sum())
    m["HKB5"]   = g["PRECTOTCORR"].transform(lambda s: hari_kering_berturut(s, ambang=5.0))
    return m

met2 = tambah_fitur_met(met)

# ---------- zonasi: HDBSCAN eom + penugasan seragam 5 km ----------
ZONASI = dict(selection="eom", mcs=150)      # alternatif: dict(selection="leaf")
R_SERAGAM = 5.0

h = hs[["tanggal", "latitude", "longitude"]].copy()
mt = (h["tanggal"] <= BATAS_TRAIN).values
lab_tr, lab_te = zona_seragam(zona_hdbscan, R_SERAGAM)(h, mt, **ZONASI)
h["zona"] = -1
h.loc[mt, "zona"] = lab_tr
h.loc[~mt, "zona"] = lab_te

htr = h.loc[mt]
htr = htr[htr["zona"] >= 0]
zi = htr.groupby("zona").agg(lat=("latitude", "mean"), lon=("longitude", "mean"))
st_rad = np.radians(np.array(list(STASIUN.values())))
JARAK_ST = haversine_distances(np.radians(zi[["lat", "lon"]].values), st_rad) * 6371   # zona x stasiun
zi["jarak_stasiun_km"] = JARAK_ST.min(axis=1)

print("Zona:", len(zi), "| cakupan test: %.1f%%" % (100 * (lab_te >= 0).mean()))

In [ ]:
nama_st = list(STASIUN)
zi["stasiun"] = [nama_st[i] for i in JARAK_ST.argmin(axis=1)]
warna_st = dict(zip(nama_st, plt.cm.Set1.colors[:len(nama_st)]))

def dasar(ax, judul):
    batas.boundary.plot(ax=ax, color="k", linewidth=0.8)
    ax.set_title(judul, fontsize=10)
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

# ---------- Peta 1: penyebaran hotspot per tahun ----------
tahun = sorted(hs["tanggal"].dt.year.unique())
fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True, sharey=True)
for ax, th in zip(axes.ravel(), tahun):
    d = hs[hs["tanggal"].dt.year == th]
    batas.boundary.plot(ax=ax, color="k", linewidth=0.6)
    ax.scatter(d["longitude"], d["latitude"], s=0.3, alpha=0.25, c="tab:red")
    ax.set_title(f"{th}  (n = {len(d):,})", fontsize=10)
plt.suptitle("Sebaran hotspot VIIRS di Kalimantan Tengah per tahun", y=1.0)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/peta_hotspot_tahunan.png", dpi=200, bbox_inches="tight")
plt.show()

# ---------- Peta 2: zona HDBSCAN + stasiun + garis ke stasiun dominan ----------
fig, ax = plt.subplots(figsize=(9, 8))
dasar(ax, f"Zona HDBSCAN ({len(zi)} zona, data training) dan stasiun meteorologi")
ns = h[mt & (h["zona"] == -1).values]
dz = h[mt & (h["zona"] >= 0).values]
ax.scatter(ns["longitude"], ns["latitude"], s=0.3, c="lightgrey", label="noise")
ax.scatter(dz["longitude"], dz["latitude"], s=0.5,
           c=[warna_st[s] for s in dz["zona"].map(zi["stasiun"])])
for z, r in zi.iterrows():
    la, lo = STASIUN[r["stasiun"]]
    ax.plot([r["lon"], lo], [r["lat"], la], color=warna_st[r["stasiun"]], lw=0.6, alpha=0.6)
ax.scatter(zi["lon"], zi["lat"], s=6, c="k", zorder=4)
for nm, (la, lo) in STASIUN.items():
    ax.scatter(lo, la, marker="^", s=170, c=[warna_st[nm]], edgecolor="k", zorder=5)
    ax.text(lo + 0.06, la + 0.06, nm, fontsize=9, weight="bold")
plt.savefig(f"{SAVE_DIR}/peta_zona_stasiun.png", dpi=200, bbox_inches="tight")
plt.show()

# ---------- Peta 3: cakupan zona pada data test (titik di dalam zona vs noise) ----------
ht = h[~mt]
fig, ax = plt.subplots(figsize=(9, 8))
dasar(ax, "Cakupan zona pada hotspot 2025–2026")
a = ht[ht["zona"] == -1]; b = ht[ht["zona"] >= 0]
ax.scatter(a["longitude"], a["latitude"], s=0.3, alpha=0.3, c="tab:red", label=f"luar zona ({100*len(a)/len(ht):.0f}%)")
ax.scatter(b["longitude"], b["latitude"], s=0.3, alpha=0.3, c="tab:blue", label=f"dalam zona ({100*len(b)/len(ht):.0f}%)")
ax.legend(markerscale=15)
plt.savefig(f"{SAVE_DIR}/peta_cakupan_test.png", dpi=200, bbox_inches="tight")
plt.show()

# TAHAP 6 Panel zona-hari, target H+1, daftar fitur, dan split berbasis waktu

In [ ]:
FITUR_MET = VAR + ["CH_3d", "CH_7d", "CH_14d", "CH_30d", "CH_60d",
                   "RH_7d", "T_7d", "HKB", "HKB5", "VPD", "VPD_7d"]

def bangun_panel_v2(h, zi, jarak_st, met2, r_tetangga=30.0, p_idw=2.0):
    dates = pd.date_range(met2["tanggal"].min(), met2["tanggal"].max())
    zones = zi.index.values
    D, Z = len(dates), len(zones)
    feats = {}

    def bcast(v):       # vektor per tanggal -> matriks D x Z
        return pd.DataFrame(np.repeat(np.asarray(v)[:, None], Z, axis=1), index=dates, columns=zones)

    # --- riwayat hotspot per zona
    cnt = (h[h["zona"] >= 0].groupby(["tanggal", "zona"]).size()
           .unstack(fill_value=0).reindex(index=dates, columns=zones, fill_value=0))
    Hb = (cnt > 0).astype(float)
    feats["hs_hari_ini"] = Hb
    feats["n_hs_log"] = np.log1p(cnt)
    for w in (3, 7, 14, 30):
        feats[f"hs_{w}d"] = Hb.rolling(w, min_periods=1).sum()
    idx = np.arange(D)[:, None]
    last = pd.DataFrame(np.where(Hb.values > 0, idx, np.nan), index=dates, columns=zones).ffill()
    feats["hari_sejak_hs"] = pd.DataFrame(idx - last.values, index=dates, columns=zones).fillna(90).clip(upper=90)

    # --- tetangga (radius r_tetangga km) dan aktivitas provinsi
    dz = haversine_distances(np.radians(zi[["lat", "lon"]].values)) * 6371
    A = ((dz <= r_tetangga) & (dz > 0)).astype(float)
    nb = pd.DataFrame((Hb.values @ A.T) / np.maximum(A.sum(axis=1), 1.0), index=dates, columns=zones)
    feats["nb_hari_ini"] = nb
    feats["nb_7d"] = nb.rolling(7, min_periods=1).mean()
    prov = Hb.mean(axis=1)
    feats["prov_aktif"] = bcast(prov.values)
    feats["prov_aktif_7d"] = bcast(prov.rolling(7, min_periods=1).mean().values)

    # --- cuaca: IDW dari stasiun ke titik pusat zona
    nama_st = list(STASIUN)
    Wt = 1.0 / np.maximum(jarak_st, 1.0) ** p_idw
    Wt = Wt / Wt.sum(axis=1, keepdims=True)                      # zona x stasiun
    for f in FITUR_MET:
        M = (met2.pivot(index="tanggal", columns="lokasi", values=f)
                  .reindex(index=dates, columns=nama_st).interpolate(limit_direction="both"))
        feats[f] = pd.DataFrame(M.values @ Wt.T, index=dates, columns=zones)

    # --- musim
    doy = dates.dayofyear.values
    feats["musim_sin"] = bcast(np.sin(2 * np.pi * doy / 365.25))
    feats["musim_cos"] = bcast(np.cos(2 * np.pi * doy / 365.25))

    # --- rakit (urutan: per tanggal, lalu per zona)
    data = {"tanggal": np.repeat(dates.values, Z), "zona": np.tile(zones, D),
            "target": Hb.shift(-1).values.ravel()}
    for k, v in feats.items():
        data[k] = v.values.ravel()
    for k in ("lat", "lon", "jarak_stasiun_km"):
        data[k] = np.tile(zi[k].values, D)
    pn = pd.DataFrame(data).dropna(subset=["target"])
    pn["target"] = pn["target"].astype(int)
    return pn.reset_index(drop=True)

pn = bangun_panel_v2(h, zi, JARAK_ST, met2)
print(pn.shape, "| positif: %.2f%%" % (100 * pn["target"].mean()))

# ---------- kelompok fitur ----------
F_A = VAR + ["CH_3d", "CH_7d", "CH_14d", "RH_7d", "T_7d", "HKB"] + \
      ["lat", "lon", "jarak_stasiun_km", "hs_hari_ini", "hs_7d"]
F_CUACA = ["CH_30d", "CH_60d", "HKB5", "VPD", "VPD_7d"]
F_HS    = ["n_hs_log", "hs_3d", "hs_14d", "hs_30d", "hari_sejak_hs"]
F_NB    = ["nb_hari_ini", "nb_7d", "prov_aktif", "prov_aktif_7d"]
F_MUSIM = ["musim_sin", "musim_cos"]
FITUR_ALL = F_A + F_CUACA + F_HS + F_NB + F_MUSIM

# ---------- split: 2024-12-31 dibuang dari train (targetnya jatuh di 2025) ----------
T0 = pd.Timestamp(BATAS_TRAIN)
tr_full = pn[pn["tanggal"] < T0].reset_index(drop=True)
te      = pn[pn["tanggal"] > T0].reset_index(drop=True)
X_full, y_full = tr_full[FITUR_ALL], tr_full["target"]

# CV per tahun: fold 1 validasi 2023 (tahun api), fold 2 validasi 2024
yr = tr_full["tanggal"].dt.year.values
folds = [(np.where(yr <= 2022)[0], np.where(yr == 2023)[0]),
         (np.where(yr <= 2023)[0], np.where(yr == 2024)[0])]

In [ ]:
FITUR_PAKAI = FITUR_ALL     # ganti setelah melihat hasil ablasi
F = FITUR_PAKAI

# TAHAP 7 Ablasi kelompok fitur

In [ ]:
def cv_tahun(cols):
    skor = []
    for tr, va in folds:
        rf, xgb = latih(X_full.iloc[tr][cols], y_full.iloc[tr], "tanpa")
        p = proba_ens(rf, xgb, X_full.iloc[va][cols])
        skor.append(average_precision_score(y_full.iloc[va], p))
    return skor

GRUP = {"A. setara fitur lama":              F_A,
        "B. + cuaca baru (VPD, hujan 30/60d)": F_A + F_CUACA,
        "C. + riwayat hotspot kaya":         F_A + F_CUACA + F_HS,
        "D. + tetangga dan provinsi":        F_A + F_CUACA + F_HS + F_NB,
        "E. + musim":                        FITUR_ALL}

def jalankan_ablasi():
    rows = []
    for nm, cols in GRUP.items():
        s = cv_tahun(cols)
        rows.append({"Kelompok": nm, "PR-AUC val 2023": s[0], "PR-AUC val 2024": s[1],
                     "Rata-rata": np.mean(s)})
    return pd.DataFrame(rows).set_index("Kelompok")

abl = cached("ablasi_fitur_v2", jalankan_ablasi)
print(abl.round(4))

# TAHAP 8 Hyperparameter tuning RF dan XGBoost (RandomizedSearchCV)

In [ ]:
rf_param2 = {"n_estimators": randint(150, 400), "max_depth": randint(3, 16),
             "min_samples_split": randint(2, 30), "min_samples_leaf": randint(1, 20),
             "max_features": ["sqrt", 0.3, 0.5], "max_samples": [0.3, 0.5, 0.8]}
xgb_param2 = {"n_estimators": randint(100, 500), "max_depth": randint(2, 8),
              "learning_rate": loguniform(0.005, 0.1), "subsample": uniform(0.6, 0.4),
              "colsample_bytree": uniform(0.5, 0.5), "min_child_weight": randint(1, 20),
              "gamma": uniform(0, 1), "reg_lambda": loguniform(0.5, 20)}

def jalankan_tuning():
    rf_s = RandomizedSearchCV(RandomForestClassifier(random_state=RS, n_jobs=1),
                              rf_param2, n_iter=15, cv=folds, scoring="average_precision",
                              random_state=RS, n_jobs=-1, verbose=1)
    rf_s.fit(X_full[FITUR_PAKAI], y_full)
    xgb_s = RandomizedSearchCV(XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                             tree_method="hist", random_state=RS, n_jobs=1),
                               xgb_param2, n_iter=15, cv=folds, scoring="average_precision",
                               random_state=RS, n_jobs=-1, verbose=1)
    xgb_s.fit(X_full[FITUR_PAKAI], y_full)
    return {"RF_PARAMS": rf_s.best_params_, "XGB_PARAMS": xgb_s.best_params_,
            "rf_skor": rf_s.best_score_, "xgb_skor": xgb_s.best_score_}

PARAM_V2 = {'RF_PARAMS': {'max_depth': 9, 'max_features': 'sqrt', 'max_samples': 0.8, 'min_samples_leaf': 5, 'min_samples_split': 20, 'n_estimators': 284}, 'XGB_PARAMS': {'colsample_bytree': 0.569746930326021, 'gamma': 0.29214464853521815, 'learning_rate': 0.014983654548550792, 'max_depth': 7, 'min_child_weight': 15, 'n_estimators': 150, 'reg_lambda': 2.0497318936355757, 'subsample': 0.9932923543227152}}

if PARAM_V2 is not None:
    tun2 = PARAM_V2                                  # cepat, tanpa tuning
else:
    tun2 = cached("tuning_v2", jalankan_tuning)      # tuning (di-cache)

RF_PARAMS, XGB_PARAMS = tun2["RF_PARAMS"], tun2["XGB_PARAMS"]

def bersih(d):
    return {k: (float(v) if isinstance(v, (float, np.floating)) else
                int(v) if isinstance(v, (int, np.integer)) else v) for k, v in d.items()}
print("PARAM_V2 =", {"RF_PARAMS": bersih(RF_PARAMS), "XGB_PARAMS": bersih(XGB_PARAMS)})

# TAHAP 9 Penanganan imbalance data (class weight dan undersampling)

In [ ]:
# ================= TAHAP 9: Penanganan imbalance (tanpa SMOTE) =================
# [UNDERSAMPLING]: Ini adalah definisi kelas untuk skenario Undersampling.
# Menggunakan teknik UnderBagging (membuat K model dari sampel kelas negatif yang di-downsample).
class UnderBagging:
    """K model, masing-masing dilatih pada subset negatif acak; probabilitas dirata-rakakan."""
    def __init__(self, buat_model, K=10, rasio=1.0, seed=RS):
        self.buat_model, self.K, self.rasio, self.seed = buat_model, K, rasio, seed
    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        yv = np.asarray(y)
        pos, neg = np.where(yv == 1)[0], np.where(yv == 0)[0]
        n_neg = min(len(neg), int(len(pos) * self.rasio))
        self.models_ = []
        for k in range(self.K):
            idx = np.concatenate([pos, rng.choice(neg, n_neg, replace=False)])
            self.models_.append(self.buat_model(k).fit(X.iloc[idx], yv[idx]))
        return self
    def predict_proba(self, X):
        p = np.mean([m.predict_proba(X)[:, 1] for m in self.models_], axis=0)
        return np.column_stack([1 - p, p])
    @property
    def feature_importances_(self):
        return np.mean([m.feature_importances_ for m in self.models_], axis=0)

K_UNDER = 10

# [BOBOT CLASS WEIGHT / TANPA]: Fungsi pembantu untuk membuat model Random Forest.
# Parameter 'cw' akan diisi jika menggunakan class_weight, dan bernilai None jika 'tanpa'.
def buat_rf(seed=RS, cw=None, n_est=None):
    p = dict(RF_PARAMS)
    if n_est: p["n_estimators"] = n_est
    return RandomForestClassifier(**p, class_weight=cw, random_state=seed, n_jobs=-1)

# [BOBOT CLASS WEIGHT / TANPA]: Fungsi pembantu untuk membuat model XGBoost.
# Parameter 'spw' (scale_pos_weight) akan diisi jika menggunakan class_weight, dan bernilai 1.0 jika 'tanpa'.
def buat_xgb(seed=RS, spw=1.0):
    return XGBClassifier(**XGB_PARAMS, objective="binary:logistic", eval_metric="logloss",
                         scale_pos_weight=spw, tree_method="hist", random_state=seed, n_jobs=-1)

# PENGATURAN SKENARIO UTAMA
def latih(X, y, skenario="tanpa"):
    # -------------------------------------------------------------------------
    # 1. SKENARIO "TANPA" (Baseline / Tanpa Penanganan Imbalance)
    # Model berjalan apa adanya menggunakan hyperparameter standar hasil tuning.
    # -------------------------------------------------------------------------
    if skenario == "tanpa":
        return buat_rf().fit(X, y), buat_xgb().fit(X, y)

    # -------------------------------------------------------------------------
    # 2. SKENARIO "CLASS_WEIGHT" (Penanganan Menggunakan Bobot Kelas)
    # RF menggunakan penyeimbang internal "balanced_subsample".
    # XGB menggunakan rumus rasio jumlah kelas negatif dibagi kelas positif (spw).
    # -------------------------------------------------------------------------
    if skenario == "class_weight":
        spw = (y == 0).sum() / (y == 1).sum()
        return buat_rf(cw="balanced_subsample").fit(X, y), buat_xgb(spw=spw).fit(X, y)

    # -------------------------------------------------------------------------
    # 3. SKENARIO "UNDERSAMPLING" (Penanganan Menggunakan Undersampling)
    # Model dilatih menggunakan kelas UnderBagging yang didefinisikan di atas.
    # -------------------------------------------------------------------------
    if skenario == "undersampling":
        n_rf = max(50, RF_PARAMS["n_estimators"] // K_UNDER)
        rf = UnderBagging(lambda k: buat_rf(RS + k, n_est=n_rf), K=K_UNDER).fit(X, y)
        xgb = UnderBagging(lambda k: buat_xgb(RS + k), K=K_UNDER).fit(X, y)
        return rf, xgb

# Ketiga skenario di atas dimasukkan ke dalam list untuk diuji secara bergantian (Looping)
SKENARIO = ["tanpa", "class_weight", "undersampling"]
MODEL = ["RF", "XGB", "Soft Voting"]

def jalankan_imbalance():
    Xte_, yte_ = te[F], te["target"]
    cv_rows, test_rows = [], []
    for sk in SKENARIO:
        oof_p = {m: [] for m in MODEL}; oof_y = []; skor = {m: [] for m in MODEL}
        for tr, va in folds:
            rf, xgb = latih(X_full.iloc[tr][F], y_full.iloc[tr], sk)
            Xv_, yv_ = X_full.iloc[va][F], y_full.iloc[va]
            ps = {"RF": rf.predict_proba(Xv_)[:, 1], "XGB": xgb.predict_proba(Xv_)[:, 1],
                  "Soft Voting": proba_ens(rf, xgb, Xv_)}
            for m in MODEL:
                oof_p[m].append(ps[m]); skor[m].append(average_precision_score(yv_, ps[m]))
            oof_y.append(yv_.values)
        oof_y = np.concatenate(oof_y)
        thr = {m: thr_optimal(oof_y, np.concatenate(oof_p[m])) for m in MODEL}
        for m in MODEL:
            cv_rows.append({"Skenario": sk, "Model": m, "PR-AUC val 2023": skor[m][0],
                            "PR-AUC val 2024": skor[m][1], "Rata-rata": np.mean(skor[m])})
        # laporan test (HANYA pelaporan, tidak dipakai untuk memilih)
        rf, xgb = latih(X_full[F], y_full, sk)
        pt = {"RF": rf.predict_proba(Xte_)[:, 1], "XGB": xgb.predict_proba(Xte_)[:, 1],
              "Soft Voting": proba_ens(rf, xgb, Xte_)}
        for m in MODEL:
            test_rows.append(evaluasi(m, yte_, pt[m], thr=thr[m], Skenario=sk, Threshold=round(thr[m], 3)))
    return {"cv": pd.DataFrame(cv_rows).set_index(["Skenario", "Model"]),
            "test": pd.DataFrame(test_rows).set_index(["Skenario", "Model"])}

imb = cached("imbalance_v2", jalankan_imbalance)
print("Seleksi (CV per tahun, PR-AUC):"); print(imb["cv"].round(4))
print("\nLaporan di test (threshold dari OOF tiap skenario):"); print(imb["test"].round(4))

# aturan seleksi (ditetapkan sebelum melihat hasil): skenario dengan PR-AUC CV tertinggi pada
# Soft Voting, kecuali selisihnya dari 'tanpa' < 0.01 -> pilih yang paling sederhana ('tanpa')
sv = imb["cv"].xs("Soft Voting", level="Model")["Rata-rata"]
SK_TERBAIK = sv.idxmax()
if sv.max() - sv["tanpa"] < 0.01:
    SK_TERBAIK = "tanpa"
print("\nSkenario terpilih:", SK_TERBAIK)